# pytorch 保存和加载通用检查点
保存和加载通用检查点可以用于推理或者恢复训练，让我们从上次打断的地方重新开始。
当我们保存通用检查点的时候，需要不存的不仅仅是模型的state_dict，同时也要保存优化器的state_dict。
因为后者包含了在训练过程中更新的缓存区和参数。
根据自己的算法，可能还需要保存离开时的epoch，最新记录的损失等等

我们将这些检查点保存在一个字典中，使用 torch.save来序列化这个字典，一般使用.tar作为文件拓展名来管理保存这些检查点。

## 导入必要的库

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 定义和初始化网络

In [6]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
net = Net()
print(net)





Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


## 初始化优化器

In [9]:
optimizer = optim.SGD(net.parameters(), lr = 0.001, momentum = 0.9)

# 保存检查点

In [12]:
EPOCH = 5
PATH = 'model.pt'
LOSS = 0.4

torch.save({
    'epoch' : EPOCH,
    'model_state_dict' : net.state_dict(),
    'optimizer_state_dict' : optimizer.state_dict(),
    'loss': LOSS}, PATH
)

## 加载通用检查点

In [14]:
model = Net()
Optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum = 0.9)

check_point = torch.load(PATH)
model.load_state_dict(check_point['model_state_dict'])
Optimizer.load_state_dict(check_point['optimizer_state_dict'])
epoch = check_point['epoch']
loss = check_point['loss']

model.eval



<bound method Module.eval of Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)>